**Description of this Notebook:** This notebook prepares the raw ZüriWieNeu report data for later spatial analysis. The main objective is to inspect, clean, and structure the original CSV file so that it can be used reliably in the following notebooks. This is an important first step because the later spatial analysis depends on this preparation.

First, the raw ZüriWieNeu CSV file is loaded and inspected. The dataset contains individual infrastructure reports from the city of Zurich, including information such as the service_request_id, submission date, update date, coordinates, service code (category), status, title, and description. The metadata are used to understand the meaning of the most important columns and to decide which attributes are relevant for the project.

Second, the dataset is cleaned by checking data types, missing values, duplicate records, and unnecessary columns. The service_request_id column is used as the stable unique identifier for each report, while objectid is not used for the analysis because it is only a system identifier. The date columns are converted into datetime format, and the coordinate columns "e" and "n" are kept because they represent the report locations in the Swiss coordinate reference system EPSG:2056, because they are in meters.

Third, additional time variables are created from the requested_datetime column. These include year_requested, month_requested, and progressing_time_days, which will later make it possible to analyse how the number of reports changes over time. Optional fields such as title, detail, and service_notice are kept only as descriptive information and are not required for the main spatial analysis.

By the end of this notebook, the raw ZüriWieNeu CSV file has been transformed into a cleaner and more structured dataset and safed in the dataproccessed folder as a new CSV.

Expected output:
- a cleaned ZüriWieNeu report dataset
- stable report IDs based on service_request_id
- valid coordinate columns e and n in EPSG:2056
- additional time columns for later temporal analysis
- a processed CSV file that can be used in the spatialjoin notebook

Firstly i import the necessary libary for this notebook. I need pandas to import and clean the CSV Dataset of the Reports.

In [1]:
import pandas as pd

The reports CSV is imported into the variable reports. It is in a relative path (..).

In [2]:
reports = pd.read_csv("../data/rawdata/data/zueriwieneu_data.csv")
#looking at the data first
reports.head(2)


,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
0,1,1,2013-03-14T15:16:15,2013-04-04T07:25:05,2013-04-12T07:59:30,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (2678968 1247548)
1,2,2,2013-03-14T15:17:57,2013-03-26T14:05:05,2013-04-12T08:00:22,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (2680746 1249916)


In [3]:
#get an overview over the attributes and how many entries, what type etc
reports.info()

<class 'pandas.DataFrame'>
RangeIndex: 72606 entries, 0 to 72605
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   objectid              72606 non-null  int64
 1   service_request_id    72606 non-null  int64
 2   requested_datetime    72606 non-null  str  
 3   agency_sent_datetime  71785 non-null  str  
 4   updated_datetime      72606 non-null  str  
 5   e                     72606 non-null  int64
 6   n                     72606 non-null  int64
 7   service_code          72606 non-null  str  
 8   service_name          72606 non-null  str  
 9   status                72606 non-null  str  
 10  userid                72606 non-null  int64
 11  title                 72604 non-null  str  
 12  detail                72604 non-null  str  
 13  media_url             49971 non-null  str  
 14  interface_used        72606 non-null  str  
 15  service_notice        71750 non-null  str  
 16  description    

In [4]:
#making a copy, so i dont change the original values while cleaning the data
reports_clean = reports.copy()
#checking if the service_request_id unique is and useful as a index
reports["service_request_id"].is_unique

True

In [5]:
#dropping the object id, because it's not stable over time said by the meta data and setting service request id as unice identifier as indexing
reports_clean = reports_clean.drop(columns=["objectid"])

reports_clean = reports_clean.set_index("service_request_id")

reports_clean.head(3)

,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
service_request_id,,,,,,,,,,,,,,,,,
1,2013-03-14T15:16:15,2013-04-04T07:25:05,2013-04-12T07:59:30,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (2678968 1247548)
2,2013-03-14T15:17:57,2013-03-26T14:05:05,2013-04-12T08:00:22,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (2680746 1249916)
4,2013-03-15T09:14:16,2013-03-15T09:55:05,2013-04-12T08:08:10,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (2684605 1251431)


In [6]:
#cleaning the names, so that they are following the same writein, so it makes it easier to write code later
reports_clean.columns = (
    reports_clean.columns
    .str.lower()
    .str.strip()
)

In [7]:
#check if relevant columns have NaN
print(reports_clean["requested_datetime"].hasnans)

False


In [8]:
print(reports_clean["updated_datetime"].hasnans)

False


In [9]:
print(reports_clean["e"].hasnans)

False


In [10]:
print(reports_clean["n"].hasnans)

False


In [11]:
print(reports_clean["service_name"].hasnans)

False


In [12]:
print(reports_clean["detail"].hasnans)

True


Okay, so detected some NaN in detail, but because its only more detail and not the service category it's not that important and i wouldnt delete the whole input but i can clean it up

In [13]:
#now the Nan values are just empty and not missing anymore. 
reports_clean["detail"] = reports_clean["detail"].fillna("")

print(reports_clean["detail"].hasnans)

False


In [14]:
print(reports_clean["status"].hasnans)

False


Okay. Now i have to convert the dates, because there are originaly in string data types, but when i want to analyse movement over time a date data type is necessary.

In [15]:
#so i dont need to format every date column alone, its easier to create a list and iterate through a list
date_cols = [
    "requested_datetime",
    "agency_sent_datetime",
    "updated_datetime"
]

for col in date_cols:
    reports_clean[col] = pd.to_datetime(
        reports_clean[col],
        format="%Y-%m-%dT%H:%M:%S",
    )
    
#check if its right now
reports_clean[date_cols].dtypes

requested_datetime      datetime64[us]
agency_sent_datetime    datetime64[us]
updated_datetime        datetime64[us]
dtype: object

Now it's time to decide which columns are useful for my analysis , why i chose only these columns is visible in my report

In [16]:
#create a list with the columns i want to keep in my report instead of dropping the colunns i want to remove makes it less messier
keep_cols = [
    "requested_datetime",
    "updated_datetime",
    "e",
    "n",
    "service_code",
    "status",
    "title",
    "detail",
    "service_notice"
]

reports_clean = reports_clean[keep_cols]
reports_clean.head(3)

,requested_datetime,updated_datetime,e,n,service_code,status,title,detail,service_notice
service_request_id,,,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,1247548,Strasse/Trottoir/Platz,fixed - council,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,Diese Reparatur wird von uns in den kommenden ...
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,1249916,Strasse/Trottoir/Platz,fixed - council,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,Diese Reparatur wird von uns in den kommenden ...
4,2013-03-15 09:14:16,2013-04-12 08:08:10,2684605,1251431,Strasse/Trottoir/Platz,fixed - council,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,Diese Reparatur wird von uns in den kommenden ...


Now its useful to add  some columns for time analysis, the year, the month and weekday aswell as the processing time is interesting

In [17]:
reports_clean["year_requested"] = reports_clean["requested_datetime"].dt.year

reports_clean["month_requested"] = reports_clean["requested_datetime"].dt.month

reports_clean["weekday_requested"] = reports_clean["requested_datetime"].dt.day_name()

#create a column for the processing time in days, so we ave the direct proccessing time in the column
reports_clean["processing_time_days"] = (
    reports_clean["updated_datetime"] - reports_clean["requested_datetime"]
).dt.total_seconds() / 86400
reports_clean.head(3)

,requested_datetime,updated_datetime,e,n,service_code,status,title,detail,service_notice,year_requested,month_requested,weekday_requested,processing_time_days
service_request_id,,,,,,,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,1247548,Strasse/Trottoir/Platz,fixed - council,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,Diese Reparatur wird von uns in den kommenden ...,2013,3,Thursday,28.696701
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,1249916,Strasse/Trottoir/Platz,fixed - council,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,Diese Reparatur wird von uns in den kommenden ...,2013,3,Thursday,28.696123
4,2013-03-15 09:14:16,2013-04-12 08:08:10,2684605,1251431,Strasse/Trottoir/Platz,fixed - council,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,Diese Reparatur wird von uns in den kommenden ...,2013,3,Friday,27.954097


I save my cleaned csv now in proccesed data folder

In [18]:
reports_clean.to_csv("../data/processeddata/zueriwieneu_cleaned.csv", index=True)